In [14]:
import pandas as pd
import numpy as np

# Adjust the path to match your exact file name in data/raw/
file_path = "../data/raw/marketing_AB.csv"

df = pd.read_csv(file_path)

In [15]:
# 1. Preview the first 5 records
print("\n--- SAMPLE RECORDS ---")
print(df.head())


--- SAMPLE RECORDS ---
   Unnamed: 0  user id test group  converted  total ads most ads day  \
0           0  1069124         ad      False        130       Monday   
1           1  1119715         ad      False         93      Tuesday   
2           2  1144181         ad      False         21      Tuesday   
3           3  1435133         ad      False        355      Tuesday   
4           4  1015700         ad      False        276       Friday   

   most ads hour  
0             20  
1             22  
2             18  
3             10  
4             14  


In [16]:
# Structural dimensions & column inspection
print("--- SHAPE & COLUMNS ---")
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")
print("\nColumn Information:")
df.info()

--- SHAPE & COLUMNS ---
Total Rows: 588,101
Total Columns: 7

Column Information:
<class 'pandas.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   Unnamed: 0     588101 non-null  int64
 1   user id        588101 non-null  int64
 2   test group     588101 non-null  str  
 3   converted      588101 non-null  bool 
 4   total ads      588101 non-null  int64
 5   most ads day   588101 non-null  str  
 6   most ads hour  588101 non-null  int64
dtypes: bool(1), int64(4), str(2)
memory usage: 27.5 MB


In [17]:
# Check for missing values
print("\n--- MISSING VALUES PER COLUMN ---")
print(df.isnull().sum())

# Check for duplicate rows and unique users
print("\n--- USER IDENTIFIER INTEGRITY ---")
# Adjust column name if it differs slightly in CSV file (e.g., 'user id' vs 'user_id')
user_col = [col for col in df.columns if "user" in col.lower()][0]
total_users = df.shape[0]
unique_users = df[user_col].nunique()
print(f"Identified User Column: '{user_col}'")
print(f"Unique Users: {unique_users:,}")
print(f"Duplicate User IDs: {total_users - unique_users:,}")



--- MISSING VALUES PER COLUMN ---
Unnamed: 0       0
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64

--- USER IDENTIFIER INTEGRITY ---
Identified User Column: 'user id'
Unique Users: 588,101
Duplicate User IDs: 0


In [18]:
# Group distribution check
group_col = [col for col in df.columns if "group" in col.lower()][0]
print(f"\n--- SPLIT BY '{group_col}' ---")
print(df[group_col].value_counts(dropna=False))
print("\nPercentage breakdown:")
print(df[group_col].value_counts(normalize=True, dropna=False) * 100)


--- SPLIT BY 'test group' ---
test group
ad     564577
psa     23524
Name: count, dtype: int64

Percentage breakdown:
test group
ad     96.000007
psa     3.999993
Name: proportion, dtype: float64


In [19]:
# Inspect categorical columns for unexpected values
print("--- UNIQUE VALUES: most ads day ---")
print(df['most ads day'].value_counts())

print("\n--- UNIQUE VALUES: most ads hour ---")
print(sorted(df['most ads hour'].unique()))

print("\n--- BASELINE CONVERSION RATES BY GROUP ---")
conversion_summary = df.groupby('test group')['converted'].agg(
    total_users='count',
    conversions='sum',
    conversion_rate='mean'
)
conversion_summary['conversion_rate_pct'] = conversion_summary['conversion_rate'] * 100
print(conversion_summary)

--- UNIQUE VALUES: most ads day ---
most ads day
Friday       92608
Monday       87073
Sunday       85391
Thursday     82982
Saturday     81660
Wednesday    80908
Tuesday      77479
Name: count, dtype: int64

--- UNIQUE VALUES: most ads hour ---
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]

--- BASELINE CONVERSION RATES BY GROUP ---
            total_users  conversions  conversion_rate  conversion_rate_pct
test group                                                                
ad               564577        14423         0.025547             2.554656
psa               23524          420         0.017854             1.785411


In [20]:
# Inspect exposure distribution (total ads)
print("\n--- DISTRIBUTION: total ads ---")
print(df['total ads'].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

# Check if any user saw 0 ads
print(f"\nUsers with 0 total ads: {(df['total ads'] == 0).sum()}")


--- DISTRIBUTION: total ads ---
count    588101.000000
mean         24.820876
std          43.715181
min           1.000000
25%           4.000000
50%          13.000000
75%          27.000000
90%          57.000000
95%          88.000000
99%         202.000000
max        2065.000000
Name: total ads, dtype: float64

Users with 0 total ads: 0


In [22]:
# Create processed directory 
import os
os.makedirs("../data/processed", exist_ok=True)

# Standardize DataFrame
df_clean = df.drop(columns=['Unnamed: 0']).copy()
df_clean.columns = [col.strip().replace(' ', '_') for col in df_clean.columns]
df_clean['converted'] = df_clean['converted'].astype(int)

print("\n--- CLEANED SCHEMA PREVIEW ---")
print(df_clean.info())
print(df_clean.head(3))

# Save processed version for SQL and Tableau pipelines
df_clean.to_csv("../data/processed/marketing_cleaned.csv", index=False)
print("\nCleaned dataset exported to data/processed/marketing_cleaned.csv")


--- CLEANED SCHEMA PREVIEW ---
<class 'pandas.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   user_id        588101 non-null  int64
 1   test_group     588101 non-null  str  
 2   converted      588101 non-null  int64
 3   total_ads      588101 non-null  int64
 4   most_ads_day   588101 non-null  str  
 5   most_ads_hour  588101 non-null  int64
dtypes: int64(4), str(2)
memory usage: 26.9 MB
None
   user_id test_group  converted  total_ads most_ads_day  most_ads_hour
0  1069124         ad          0        130       Monday             20
1  1119715         ad          0         93      Tuesday             22
2  1144181         ad          0         21      Tuesday             18

Cleaned dataset exported to data/processed/marketing_cleaned.csv
